In [1]:
import numpy as np
import operator
from collections import Counter
import pandas as pd

class CART:
    def __init__(self,pruning = None):
        self.tree = None
        self.pruning = pruning
        """
        CART决策树

        参数:
        - pruning: 剪枝方法,None表示不剪枝,其他值表示剪枝方法,目前支持'post'剪枝方法
        """

    def fit(self, dataset_train, dataset_test, features):
        """
        训练模型
        
        参数:
        - dataset_train: 训练数据集
        - dataset_test: 测试数据集
        - features: 特征列表 
        """
        self.features = features
        self.tree = self.createTree(dataset_train, features)

    def createTree(self, Dataset, features):
        """ 
        创建决策树
        
        参数:
        - Dataset: 训练数据集
        - features: 特征列表
        
        返回:
        - 决策树
        """
        # 提取标签
        classList = [example[-1] for example in Dataset]

        # 如果所有样本的标签相同，则返回该标签
        if len(set(classList)) == 1:
            return classList[0]
        
        # 如果特征集为空，则返回出现次数最多的标签
        if len(Dataset[0]) == 1:
            return self.majorityCnt(classList)
        
        # 选择最优特征
        bestFeatureIndex, bestSplitValue = self.chooseBestFeature(Dataset)
        bestFeatureLabel = features[bestFeatureIndex]

        # 创建节点
        tree = {bestFeatureLabel: {}}
        # 使用副本避免修改原始列表
        subfeatures = features.copy()
        # 删除当前特征
        del subfeatures[bestFeatureIndex]
        # 连续特征
        if type(bestSplitValue).__name__ == 'float':
            tree[bestFeatureLabel]['<=' + str(bestSplitValue)] = self.createTree(self.splitDataSetByValue(Dataset, bestFeatureIndex, bestSplitValue, False), subfeatures)
            tree[bestFeatureLabel]['>' + str(bestSplitValue)] = self.createTree(self.splitDataSetByValue(Dataset, bestFeatureIndex, bestSplitValue, True), subfeatures)
        # 离散特征
        else:
            # 取出当前特征的取值
            featValue = [example[bestFeatureIndex] for example in Dataset]
            uniqueVals = set(featValue)
            # 遍历所有取值,开始递归
            for value in uniqueVals:
                subDataset = self.splitDataSet(Dataset, bestFeatureIndex, value)
                tree[bestFeatureLabel][value] = self.createTree(subDataset, subfeatures)
        return tree
    
    def majorityCnt(self, classList):
        """返回最多的标签"""
        # 统计标签出现的次数
        label_count = {}
        for label in classList:
            if label not in label_count:
                label_count[label] = 0
            label_count[label] += 1
        # 降序排序[(类标签,出现次数),(),()]
        sortedclassCount = sorted(label_count.items(), key=operator.itemgetter(1), reverse=True)
        # 当所有标签出现的个数相同时，返回最后一个标签(非必要，只是为了保持一致性)
        if sortedclassCount[0][1] == sortedclassCount[-1][1]:
            return sortedclassCount[-1][0]
        # 返回出现次数最多的标签
        return sortedclassCount[0][0]

    def chooseBestFeature(self,Dataset):
        """通过信息增益选择最优特征"""
        featureNum = len(Dataset[0]) - 1
        bestGini = 1.0
        bestSplitValue = 0
        bestFeatureIndex = -1

        # 遍历所有特征
        for i in range(featureNum):
            # 提取当前特征下的取值
            featureValues = [example[i] for example in Dataset]
            # 连续特征
            if type(featureValues[0]).__name__ == 'float':
                # 对特征值进行排序
                sortedFeatureValues = sorted(featureValues)
                # 计算分割值（取相邻两个取值的中点）
                splitList = []
                for j in range(len(sortedFeatureValues) - 1):
                    splitList.append((sortedFeatureValues[j] + sortedFeatureValues[j + 1]) / 2.0)
                # 遍历所有分割值,相当于做二分类
                for splitValue in splitList:
                    currentEntropy = 0.0
                    subDataset1 = self.splitDataSetByValue(Dataset, i, splitValue, True)
                    subDataset2 = self.splitDataSetByValue(Dataset, i, splitValue, False)
                    prob1 = len(subDataset1) / float(len(Dataset))
                    prob2 = len(subDataset2) / float(len(Dataset))
                    currentEntropy  = prob1 * self.calculateGini(subDataset1) + prob2 * self.calculateGini(subDataset2)
                    # 比较Gini指数，数值越小，纯度越高
                    if (currentGini < bestGini):
                        bestGini = currentGini
                        bestFeatureIndex = i
                        bestSplitValue = splitValue
            # 离散特征
            else:
                uniqueValues = set(featureValues)
                currentGini = 0.0
                # 遍历所有取值,计算Gini指数
                for value in uniqueValues:
                    subDataset = self.splitDataSet(Dataset, i, value)
                    prob = len(subDataset) / len(Dataset)
                    currentGini += prob * self.calculateGini(subDataset)
                # 比较Gini指数
                if (currentGini < bestGini):
                    bestGini = currentGini
                    bestFeatureIndex = i
                    bestSplitValue = None
        
        return bestFeatureIndex, bestSplitValue

    def calculateGini(self, Dataset):
        """计算Gini指数,公式(4.5)"""
        sample_num = len(Dataset)
        # 统计标签出现的次数
        label_count = {}
        for featVec in Dataset:
            label = featVec[-1]
            if label not in label_count:
                label_count[label] = 0
            label_count[label] += 1
        
        # 计算信息熵
        Gini_index = 1.0
        for count in label_count.values():
            prob = float(count) / sample_num
            Gini_index -= prob ** 2

        return Gini_index

    def splitDataSet(self,Dataset, axis, val):
        '''
        根据特征索引i和离散特征值value将数据集切分

        参数:
        - Dataset: 训练数据集
        - axis: 特征索引
        - val: 特征值

        返回:
        - 切分后的子集
        '''
        subDataset = []
        # 遍历每一行
        for featVec in Dataset:
            if featVec[axis] == val:
                reducedFeature = featVec[:axis]
                reducedFeature.extend(featVec[axis + 1:])
                subDataset.append(reducedFeature)
        return subDataset

    def splitDataSetByValue(self, Dataset, axis, val, isAbove):
        '''
        根据特征索引i和连续特征值value将数据集切分

        参数:
        - Dataset: 训练数据集
        - axis: 特征索引
        - val: 特征值
        - isAbove: True表示大于value,False表示小于等于value

        返回:
        - 切分后的子集
        '''
        subDataset = []
        # 遍历每一行
        for featVec in Dataset:
            if isAbove and featVec[axis] > val:
                reducedFeature = featVec[:axis]
                reducedFeature.extend(featVec[axis + 1:])
                subDataset.append(reducedFeature)
            elif not isAbove and featVec[axis] <= val:
                reducedFeature = featVec[:axis]
                reducedFeature.extend(featVec[axis + 1:])
                subDataset.append(reducedFeature)
        return subDataset

    def predict(self, inputTree,features, testVec):
        '''
        预测测试数据集

        参数:
        - inputTree: 训练好的决策树
        - features: 特征列表
        - testVec: 测试数据集

        返回:
        - 预测结果
        '''
        # 如果当前树只含一个节点，则直接返回该节点的值
        if not isinstance(inputTree, dict):
            return inputTree

        # 提取当前节点(每个决策树节点只有一个特征标签)
        firstStr = list(inputTree.keys())[0]
        # 提取当前节点下的子节点
        secondDict = inputTree[firstStr]
        # 获取当前节点的特征标签序号
        featureIndex = features.index(firstStr)

        # 遍历每个子节点
        for key in secondDict.keys():
            # 连续特征
            if type(key).__name__ == 'str' and ('<=' in key or '>' in key):
                # 去除字符串中的符号，取出阈值
                threshold = float(key.strip('<=').strip('>'))
                # 判断测试数据是否满足阈值
                if key.startswith('<=') and testVec[featureIndex] <= threshold:
                    childTree = secondDict[key]
                    # 判断当前是不是叶节点,如果不是，继续递归
                    if isinstance(childTree, dict):
                        return self.predict(childTree,features, testVec)
                    else:
                        return childTree
                elif key.startswith('>') and testVec[featureIndex] > threshold:
                    childTree = secondDict[key]
                    # 判断当前是不是叶节点
                    if isinstance(childTree, dict):
                        return self.predict(childTree,features, testVec)
                    else:
                        return childTree
            # 离散特征
            else:
                # 判断测试数据是否满足取值
                if testVec[featureIndex] == key:
                    childTree = secondDict[key]
                    # 判断当前是不是叶节点
                    if isinstance(childTree, dict):
                        return self.predict(childTree,features, testVec)
                    else:
                        return childTree
        return "Unknown !"
        
    def printTree(self):
        """打印决策树"""
        print(self.tree)

    def calculateAccuracy(self, dataset_test, features):
        """计算准确率"""
        correct_count = 0
        for example in dataset_test:
            label = example[-1]
            predict_label = self.predict(self.tree, features, example)
            if label == predict_label:
                correct_count += 1
        accuracy = float(correct_count) / len(dataset_test)
        return accuracy
    
#######################随机森林################################

class RandomForest:
    def __init__(self, n_estimators=10,  pruning=None):
        """
        随机森林分类器
        
        参数:
        - n_estimators: 森林中树的数量
        - pruning: 剪枝方法，None表示不剪枝，'post'表示后剪枝
        """
        self.n_estimators = n_estimators
        self.pruning = pruning
        self.trees = []  # 存储所有决策树
        self.feature_subsets = []  # 存储每棵树使用的特征子集索引
    
    def fit(self, dataset_train, dataset_test, features):
        """
        训练随机森林模型
        
        参数:
        - dataset_train: 训练数据集
        - dataset_test: 测试数据集
        - features: 特征列表 
        """
        n_features = len(features)
        
        # 确定每棵树使用的特征数为log2d
        n_sub_features = max(1, int(np.log2(n_features)))
        
        for _ in range(self.n_estimators):
            # 自助法抽取样本
            n_samples = len(dataset_train)
            bootstrap_indices = np.random.choice(int(n_samples), n_samples, replace=True)
            bootstrap_dataset = [dataset_train[i] for i in bootstrap_indices]
            
            # 随机选择特征子集
            selected_feature_indices = np.random.choice(n_features, n_sub_features, replace=False)
            selected_feature_indices.sort()
            
            # 创建特征子集对应的数据集（仅包含选择的特征）
            feature_subset_dataset_train = []
            for sample in bootstrap_dataset:
                new_sample = [sample[i] for i in selected_feature_indices]
                new_sample.append(sample[-1])  # 添加标签
                feature_subset_dataset_train.append(new_sample)
            
            feature_subset_dataset_test = []
            for sample in dataset_test:
                new_sample = [sample[i] for i in selected_feature_indices]
                new_sample.append(sample[-1])  # 添加标签
                feature_subset_dataset_test.append(new_sample)
            
            # 特征子集
            selected_features = [features[i] for i in selected_feature_indices]
            
            # 训练CART树
            cart = CART()
            cart.fit(feature_subset_dataset_train, feature_subset_dataset_test, selected_features)
            
            # 存储树和特征子集信息
            self.trees.append(cart)
            self.feature_subsets.append(selected_feature_indices)
            
    
    def predict(self, testVec):
        """
        预测单个样本
        
        参数:
        - testVec: 测试样本特征向量（不含标签）
        
        返回:
        - 预测结果
        """
        predictions = []
        for cart, feature_indices in zip(self.trees, self.feature_subsets):
            # 根据特征子集选择测试样本特征
            selected_testVec = [testVec[i] for i in feature_indices]
            prediction = cart.predict(cart.tree, cart.features, selected_testVec)
            predictions.append(prediction)
        
        # 投票决定最终预测结果
        return Counter(predictions).most_common(1)[0][0]
    
    def calculateAccuracy(self, dataset_test):
        """
        计算随机森林在测试集上的准确率
        
        参数:
        - dataset_test: 测试数据集
        
        返回:
        - 准确率
        """
        correct_count = 0
        for example in dataset_test:
            features = example[:-1]  # 提取特征（不含标签）
            label = example[-1]       # 真实标签
            prediction = self.predict(features)
            if label == prediction:
                correct_count += 1
        
        accuracy = float(correct_count) / len(dataset_test)
        return accuracy
    
    def printTree(self, tree_index=0):
        """
        打印指定索引的决策树
        
        参数:
        - tree_index: 要打印的树的索引
        """
        if tree_index < len(self.trees):
            print(f"Tree {tree_index}:")
            self.trees[tree_index].printTree()
        else:
            print(f"Error: Tree index {tree_index} out of range.")


if __name__ == '__main__':
    # 加载数据集
    df = pd.DataFrame(pd.read_csv("../Data/watermelon2.0.csv", encoding="ansi"))
    df.drop(labels=["编号"], axis=1, inplace=True)  # 删除编号这一列，inplace=True表示直接在原对象修改
    # 转化为列表
    dataset = df.values.tolist()
    # 第4，5，8，9，11，12，13行作为测试集
    dataset_test = [dataset[i-1] for i in [4,5,8,9,11,12,13]]

    # 其余作为训练集
    dataset_train = [dataset[i-1] for i in range(len(dataset)) if i not in [4,5,8,9,11,12,13]]

    # 打印数据集
    # for i in dataset:
    #     print(i)
        
    # 属性
    features = ['色泽', '根蒂', '敲声', '纹理', '脐部', '触感']
    
    # 训练
    cart = CART()
    cart.fit(dataset_train, dataset_test, features)
    random_forest = RandomForest(n_estimators=100, pruning='None')
    random_forest.fit(dataset_train, dataset_test, features)

    # 计算准确率
    acccuracy_CART = cart.calculateAccuracy(dataset_test, features)
    print("CART Accuracy:", acccuracy_CART)
    accuracy_RF = random_forest.calculateAccuracy(dataset_test)
    print("Random Forest Accuracy:", accuracy_RF)
    

CART Accuracy: 0.2857142857142857
Random Forest Accuracy: 0.8571428571428571
